# Corpus Router

Простой пайплайн для разметки корпуса по категориям PII.

Роутер **не знает про конкретные сущности** и не решает, какую сущность вставлять. Его задача только:

```text
текст -> подходящие категории / unsuitable
```

Связь `категория -> сущности` потом можно отдельно использовать в синтезаторе.


In [3]:
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mistralai import ChatMistralAI
from pydantic import BaseModel, Field

load_dotenv()


True

In [4]:
# Основные настройки
CORPUS_PATH = "../data/texts_200_500_symbols.csv"
TEXT_COLUMN = "text"
OUTPUT_PATH = "outputs/corpus_router_tagged.jsonl"

MODEL_NAME = "mistral-small-latest"
BATCH_SIZE = 20
START = 0
LIMIT = 100  # для первого теста лучше 100-300, потом можно поднять
SLEEP_SEC = 0.5

ENTITY_GROUPS = {
    "personal_documents": "личные документы, оформление, анкеты, паспортные/госуслуговые ситуации",
    "address_geographic_data": "адрес, город, регион, переезд, жилье, место проживания или нахождения",
    "personal_biographical_data": "имя, ФИО, возраст, дата рождения, личная биография, рассказ о себе или другом человеке",
    "medical_data": "здоровье, лечение, врач, клиника, диагнозы, анализы, медицинские документы",
    "financial_and_banking_identifiers": "деньги, переводы, оплата, банки, карты, счета, реквизиты, долги",
    "transport_identifiers": "машина, транспорт, ДТП, права, номера, документы на автомобиль",
    "educational_documents": "школа, вуз, учеба, диплом, аттестат, студенческие документы",
    "judicial_and_executive_identifiers": "суд, полиция, приставы, штрафы, исполнительные производства, юридические документы",
    "social_media_and_messengers": "соцсети, мессенджеры, аккаунты, личные сообщения, никнеймы, профили",
    "gaming_platforms": "игры, игровые аккаунты, игровые платформы, игровые ID, игровые ники",
    "career_platforms": "работа, резюме, вакансии, рекрутеры, профессиональные профили, поиск работы",
    "family_and_social_documents": "семья, брак, развод, дети, свидетельства, социальные документы",
    "tracking_numbers": "заказы, доставки, посылки, курьеры, почта, трекинг",
    "digital_identifiers": "логины, user id, client id, IP, технические аккаунты и цифровые идентификаторы",
}

ALL_LABELS = list(ENTITY_GROUPS)


In [5]:
class RoutedText(BaseModel):
    text_id: int = Field(description="ID текста из входного батча")
    labels: list[str] = Field(description="0-3 подходящие категории из списка ENTITY_GROUPS")
    unsuitable: bool = Field(description="True, если текст мусорный или не подходит ни под одну категорию")
    reason: str = Field(description="Короткое объяснение на русском, максимум одно предложение")


class RoutedBatch(BaseModel):
    items: list[RoutedText]


llm = ChatMistralAI(
    model=MODEL_NAME,
    temperature=0.7,
    timeout=60,
)

structured_llm = llm.with_structured_output(RoutedBatch)


In [6]:
import pandas as pd

texts = pd.read_csv(CORPUS_PATH, index_col = 0).squeeze()
texts = texts.reset_index(drop = True)
print(texts)
print(texts.info())

text_lst = texts.to_list()

0        Вот как и многие тут мечусь между жизнью и сме...
1        Я устала от жестокости мира, от глупости и жад...
2        Здравствуйте, эта тема и это сообщения ничто. ...
3        мне планируют платить пенсию 130 евро в месяц....
4        У меня букет психических расстройств, всю свою...
                               ...                        
19455    Изменила мужу. Не смогла ему признаться, но и ...
19456    не знаю стоит ли писать. в общем четыре месяца...
19457    Много чего есть рассказать. Коротко. Очень сло...
19458    я просто схожу с ума.в такие моменты хочется и...
19459    Добрый вечер. я думаю о самоубийстве, последне...
Name: text, Length: 19460, dtype: str
<class 'pandas.Series'>
RangeIndex: 19460 entries, 0 to 19459
Series name: text
Non-Null Count  Dtype
--------------  -----
19460 non-null  str  
dtypes: str(1)
memory usage: 10.3 MB
None


In [7]:
SYSTEM_PROMPT = f"""
У нас есть задача: вставлять в пользовательские тексты PII-сущности так, чтобы вставка выглядела естественно и не ломала смысл текста.

Твоя задача: для каждого текста выбрать 0-3 категории PII-сущностей, которые уместнее всего вставлять именно в этот текст.

Вход: обычный пользовательский текст.
Выход: категории сущностей, которые можно естественно встроить в этот текст небольшой правкой или коротким продолжением.

Главное:
- Не ищи ключевые слова.
- Не требуй, чтобы PII уже были в тексте.
- Не подбирай категории механически по отдельным словам.
- Оценивай весь смысл текста, его ситуацию, стиль и возможное естественное продолжение.

Как принимать решение:
1. Представь, что синтезатор может слегка дополнить текст 1-2 фразами, сохранив общий стиль автора.
2. Выбери категории, для которых такое дополнение выглядело бы правдоподобно.
3. Не выбирай категорию, если для неё пришлось бы полностью менять тему текста или вставка выглядела бы случайной.
4. Если текст пригоден для нескольких сценариев, выбери самые естественные 1-3 категории.
5. Если текст мусорный, слишком короткий, рекламный, технический, на неподходящем языке или к нему нельзя нормально добавить ни одну категорию, верни labels=[] и unsuitable=true.

Важно:
- Категории ниже являются смысловыми классами, а не списками слов для поиска.
- reason должен кратко описывать возможный сценарий дополнения, а не перечислять найденные слова.
- labels должны быть строго из списка доступных категорий.

Доступные категории:
{json.dumps(ENTITY_GROUPS, ensure_ascii=False, indent=2)}
""".strip()

def route_batch(batch: list[dict]) -> list[dict]:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(batch, ensure_ascii=False, indent=2)),
    ]
    response = structured_llm.invoke(messages)

    result_by_id = {item.text_id: item.model_dump() for item in response.items}
    rows = []
    for source in batch:
        item = result_by_id.get(source["text_id"], {
            "text_id": source["text_id"],
            "labels": [],
            "unsuitable": True,
            "reason": "Модель не вернула разметку для этого текста.",
        })

        labels = [label for label in item.get("labels", []) if label in ALL_LABELS]
        labels = labels[:3]
        unsuitable = bool(item.get("unsuitable", False)) or len(labels) == 0

        rows.append({
            "text_id": source["text_id"],
            "text": source["text"],
            "labels": labels,
            "unsuitable": unsuitable,
            "reason": item.get("reason", "").strip(),
            "model": MODEL_NAME,
        })
    return rows


In [8]:
def make_batches(text_lst: list[str], batch_size: int = BATCH_SIZE, start: int = START, limit: int | None = LIMIT):
    end = None if limit is None else start + limit
    selected = text_lst[start:end]

    for batch_start in range(0, len(selected), batch_size):
        chunk = selected[batch_start:batch_start + batch_size]
        yield [
            {"text_id": start + batch_start + i, "text": text}
            for i, text in enumerate(chunk)
        ]


batches = list(make_batches(text_lst))
print(f"Всего батчей: {len(batches)}")
print(f"Размер первого батча: {len(batches[0]) if batches else 0}")
batches[0][:2] if batches else []


Всего батчей: 5
Размер первого батча: 20


[{'text_id': 0,
  'text': 'Вот как и многие тут мечусь между жизнью и смертью. Как-то худо вяло живу в таком состоянии. Мне интересно - был ли у кого-то здесь опыт выхода из полнейшей депрессухи, когда вот это вот все пустота, одиночество и прочее съедают изнутри?'},
 {'text_id': 1,
  'text': 'Я устала от жестокости мира, от глупости и жадности людской, от перетягивания одеяла на себя, от постоянной борьбы за выживание. Хочу тихо и спокойно жить, чтоб никто не трогал и небыло нужды в борьбе за удовлетворение первичных потребностей, но этого не дано. Такое чувство что суть жизни - мучения. Хочу покоя. раз не возможен он в жизни, тогда лишь надежда на смерть. уже нет сил её ждать, таких земля как на зло долго носит. выхода другого не вижу, кроме как приблизить костлявую.'}]

In [9]:
all_rows = []
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

for i, batch in enumerate(batches, start=1):
    print(f"Batch {i}/{len(batches)}: {batch[0]['text_id']}..{batch[-1]['text_id']}")
    rows = route_batch(batch)
    all_rows.extend(rows)

    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    time.sleep(SLEEP_SEC)

print(f"Готово. Сохранено строк: {len(all_rows)}")


Batch 1/5: 0..19
Batch 2/5: 20..39
Batch 3/5: 40..59
Batch 4/5: 60..79
Batch 5/5: 80..99
Готово. Сохранено строк: 100
